# 05 · Swiss roll: experimentos de cierre

Tres experimentos con hipótesis y criterios de decisión **fijados antes de correr**. Todos usan el paquete v4, kl = 10⁻³, 10 000 pasos de Adam por modelo, `vae` entrenado desde cero (`scratch`) y por continuación en m (`warm`), y réplicas con datos y partición independientes.

| | Pregunta | Diseño | Criterio |
|---|---|---|---|
| **A** | ¿La continuación espectral despliega la carta de forma fiable? | 60×10, orden espectral, **16 réplicas** | tasa de `vae_warm` desplegado con límite inferior Clopper-Pearson (95 %) > 0.6 para todo m ≥ 8 |
| **B** | ¿El efecto viene del **orden espectral** o de la continuación en sí? | igual que A, con los modos en **orden aleatorio** (una permutación por réplica) | comparación en **m = 64 = M**: mismo conjunto de modos y misma clase de modelos en ambos órdenes, solo cambia el camino. Fisher exacto unilateral, p < 0.05 |
| **D** | ¿El umbral de m se mueve como predice el teorema de prefijo? | enrollado **30×10**: el modo de altura pasa de la posición 6–7 a la 3–4 | el oráculo cae y `vae_warm` despliega a partir de m = 3–4, no de 6–7 |

Control en B: con `scratch` en m = 64 ambos órdenes deben ser estadísticamente iguales (la permutación de las entradas no cambia la clase). Si no lo son, hay un error en el código.

Duración estimada: ~7 h en dos T4. Las configuraciones se retoman si la sesión se corta. Para una corrida larga conviene **Save Version → Save & Run All (Commit)**, que sigue en segundo plano.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo de autovectores, métrica kNN, continuación con ceros y paso AE→VAE. Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_05.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_05.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β para cada geometría

- **60×10:** factor 64, elegido por el barrido sin etiquetas del notebook 01a (6 semillas de datos, n = 3000). Si ese barrido existe en esta sesión, se lee de ahí.
- **30×10:** es otra geometría (menos vueltas, otra densidad y otra mediana de distancias), así que necesita su propio barrido con la misma regla. ~10–15 min.

In [ ]:
import time
FACTOR_60 = 64
prev = Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')
if prev.exists():
    FACTOR_60 = json.loads(prev.read_text())['selected_factor']
print('factor 60x10 =', FACTOR_60)

cfg_sweep = json.loads((ROOT / 'config_beta_sweep.json').read_text())
cfg_sweep.update(width=30.0, height=10.0, modes=[2, 3, 4, 5, 6, 8, 16, 32, 64],
                 out='/kaggle/working/beta_sweep_swiss_30x10')
SWEEP_30 = Path(cfg_sweep['out'])
if not (SWEEP_30 / 'beta_selection.json').exists():
    ACTIVE = ROOT / 'active_beta_sweep_30x10.json'
    ACTIVE.write_text(json.dumps(cfg_sweep, indent=2))
    t0 = time.time()
    r = subprocess.run([sys.executable, str(ROOT / 'beta_sweep.py'), str(ACTIVE), '--launch'],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:]); print(r.stderr[-6000:])
        for log in sorted(SWEEP_30.glob('sweep_worker*.log')):
            print('---', log.name); print(log.read_text()[-3000:])
        raise RuntimeError('Falló el barrido de beta para 30x10.')
    print(f'Barrido 30x10: {(time.time()-t0)/60:.1f} min')
sel30 = json.loads((SWEEP_30 / 'beta_selection.json').read_text())
FACTOR_30 = sel30['selected_factor']
assert FACTOR_30 is not None, 'Ningún factor admisible para 30x10: amplía la lista de factores.'
import pandas as pd
from IPython.display import display, Image
sw30 = pd.read_csv(SWEEP_30 / 'beta_sweep.csv')
cols = [c for c in ['kernel_length', 'graph_shortcut_mass', 'true_shortcut_mass', 'psi1_corr_cos_s',
                    'height_mode_position', 'expected_height_mode_position', 'oracle_val_m2',
                    'oracle_val_m3', 'oracle_val_m4', 'oracle_val_m8'] if c in sw30]
display(sw30.groupby('factor')[cols].mean().style.format('{:.3g}').set_caption(
    f'30x10 · seleccionado: factor {FACTOR_30} (banda admisible {sel30["admissible"]})'))
print('Predicción del teorema para 30x10: modo de altura en la posición 3 o 4 (par degenerado k=3 / altura).')
display(Image(filename=str(SWEEP_30 / 'beta_sweep.png')))

## Plan y estimación de tiempo

El orden de `PLAN` es la prioridad. Si el tiempo no alcanza, las configuraciones completas se omiten al re-ejecutar y las demás se retoman. Ajusta `SECONDS_PER_MODEL` con el tiempo real de la primera configuración.

In [ ]:
import math
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
COMMON = dict(epochs=2500, batch=512, min_steps=10_000, tau=1.0, kl=1e-3, resample_data=True,
              knn_k=5, unfold_ratio=10.0, restarts=1, backend='flash')
LADDER_60 = [2, 4, 8, 16, 32, 64]
LADDER_30 = [2, 3, 4, 5, 6, 8, 16, 32]
PLAN = [
    dict(tag='A_spectral', width=60., height=10., beta_factor=FACTOR_60, seeds=list(range(16)),
         modes=LADDER_60, models=['vae'], m_protocols=['scratch', 'warm'], mode_order='spectral',
         chart_decoder_epochs=10_000),      # referencia más larga una sola vez (no depende del orden)
    dict(tag='B_shuffled', width=60., height=10., beta_factor=FACTOR_60, seeds=list(range(16)),
         modes=LADDER_60, models=['vae'], m_protocols=['scratch', 'warm'], mode_order='shuffled',
         mode_order_seed=12345),
    dict(tag='D_30x10_vae', width=30., height=10., beta_factor=FACTOR_30, seeds=list(range(8)),
         modes=LADDER_30, models=['vae'], m_protocols=['scratch', 'warm'], mode_order='spectral'),
    dict(tag='D_30x10_direct', width=30., height=10., beta_factor=FACTOR_30, seeds=list(range(4)),
         modes=LADDER_30, models=['direct'], m_protocols=['scratch'], mode_order='spectral'),
]
SECONDS_PER_MODEL = 80
def modelos_por_semilla(p):
    base = [k for k in p['models'] if k != 'plainvae']
    n = len(p['modes']) * len(base)
    if 'warm' in p['m_protocols']:
        n += (len(p['modes']) - 1) * len([k for k in base if k in ('ae', 'vae', 'direct')])
    cd = p.get('chart_decoder_epochs', COMMON['epochs']) / COMMON['epochs']
    return n * COMMON['restarts'] + cd
for p in PLAN:
    p['out'] = f"/kaggle/working/results_swiss_{p['tag']}"
    p['eta_min'] = math.ceil(len(p['seeds']) / 2) * modelos_por_semilla(p) * SECONDS_PER_MODEL / 60
print(f"{'configuración':16s}{'beta':>6s}{'geom':>7s}{'orden':>10s}{'semillas':>9s}{'modelos':>9s}{'ETA min':>9s}")
for p in PLAN:
    print(f"{p['tag']:16s}{p['beta_factor']:6g}{int(p['width']):4d}x{int(p['height']):<2d}{p['mode_order']:>10s}"
          f"{len(p['seeds']):9d}{modelos_por_semilla(p):9.0f}{p['eta_min']:9.0f}")
print(f"\nTOTAL estimado: {sum(p['eta_min'] for p in PLAN)/60:.1f} h")

In [ ]:
import time
hechas, fallidas = [], []
for p in PLAN:
    out = Path(p['out'])
    if (out / 'metrics.csv').exists():
        print(f"{p['tag']:16s} ya completo, se omite"); hechas.append(p); continue
    if any(out.glob('seed_*/metrics.json')):
        print(f"{p['tag']:16s} PARCIAL en {out}: bórralo o cambia el tag"); fallidas.append((p['tag'], 'parcial')); continue
    keys = ('width', 'height', 'beta_factor', 'seeds', 'modes', 'models', 'm_protocols', 'mode_order',
            'mode_order_seed', 'chart_decoder_epochs', 'out')
    cfg = dict(BASE, **COMMON, **{k: p[k] for k in keys if k in p})
    path = ROOT / f"active_swiss_{p['tag']}.json"
    path.write_text(json.dumps(cfg, indent=2))
    t0 = time.time()
    r = subprocess.run([sys.executable, str(ROOT / 'launch.py'), str(path)], capture_output=True, text=True)
    print(f"{p['tag']:16s} {(time.time()-t0)/60:6.1f} min (ETA {p['eta_min']:.0f})", flush=True)
    if r.returncode:
        print(r.stdout[-2000:]); print(r.stderr[-3000:]); fallidas.append((p['tag'], 'falló'))
    else:
        hechas.append(p)
print('\nhechas:', [p['tag'] for p in hechas], '| problemas:', fallidas)

## Análisis

Tasas con intervalo de Clopper-Pearson al 95 %. "Desplegada" = kNN(latente → carta) ≤ 10 × el piso del estimador con la carta verdadera. Solo filas de modelos realmente entrenados (se excluye la fila duplicada del inicio de cada cadena `warm`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist, fisher_exact, mannwhitneyu

def cargar(tag):
    csv = Path(f'/kaggle/working/results_swiss_{tag}') / 'metrics.csv'
    if not csv.exists():
        return None
    d = pd.read_csv(csv)
    d['restart'] = d['restart'].fillna(0).astype(int) if 'restart' in d else 0
    return d

def entrenados(d):
    q = d[d.validation_mse_x.notna() & d.m.notna()]
    return q[q.warm_start.isna()] if 'warm_start' in q else q

def clopper_pearson(k, n, a=.05):
    lo = beta_dist.ppf(a/2, k, n-k+1) if k > 0 else 0.
    hi = beta_dist.ppf(1-a/2, k+1, n-k) if k < n else 1.
    return lo, hi

def tabla_despliegue(d, etiqueta):
    rows = []
    for (model, m), g in entrenados(d).groupby(['model', 'm']):
        if 'unfolded' not in g or g.unfolded.isna().all():
            continue
        k, n = int(g.unfolded.astype(bool).sum()), int(g.unfolded.notna().sum())
        lo, hi = clopper_pearson(k, n)
        q = g.mse_x.quantile([.25, .5, .75]).values
        rows.append(dict(corrida=etiqueta, model=model, m=int(m), desplegadas=f'{k}/{n}', tasa=k/n,
                         ic95_inf=lo, ic95_sup=hi, mse_x_mediana=q[1], mse_x_iqr=f'{q[0]:.3g}–{q[2]:.3g}',
                         knn_ratio_mediana=g.knn_chart_ratio.median()))
    return pd.DataFrame(rows)

A, B = cargar('A_spectral'), cargar('B_shuffled')
fmt = {'tasa': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'mse_x_mediana': '{:.4g}',
       'knn_ratio_mediana': '{:.3g}'}

# ---- A: fiabilidad de la continuación espectral ----
if A is not None:
    tA = tabla_despliegue(A, 'A espectral')
    display(tA.style.format(fmt).set_caption('A · 60×10, orden espectral'))
    w = tA[(tA.model == 'vae_warm') & (tA.m >= 8)]
    ok = len(w) and (w.ic95_inf > .6).all()
    print(f"Criterio A (IC inferior > 0.6 para vae_warm en todo m >= 8): {'CUMPLIDO' if ok else 'NO cumplido'}")
    cd = A[A.model == 'chart_decoder']
    print(f"decoder sobre la carta verdadera (referencia, {int(cd.optimizer_steps.iloc[0]) if 'optimizer_steps' in cd else '?'} pasos): "
          f"MSE_x = {cd.mse_x.mean():.3g} ± {cd.mse_x.std():.2g}")

# ---- B: ¿orden espectral o continuación? ----
if A is not None and B is not None:
    tB = tabla_despliegue(B, 'B aleatorio')
    display(tB.style.format(fmt).set_caption('B · 60×10, orden aleatorio de los modos'))
    oA = A[A.model == 'oracle']
    oB = B[B.model == 'oracle']
    info = pd.DataFrame({'oráculo espectral': oA.groupby('m').chart_mse_z.median(),
                         'oráculo aleatorio': oB.groupby('m').chart_mse_z.median()})
    display(info.style.format('{:.3g}').set_caption(
        'Información de cada prefijo (oráculo afín, chart MSE_z, mediana). En m = 64 ambos tienen los mismos modos.'))
    M = max(LADDER_60)
    res = []
    for model in ('vae', 'vae_warm'):
        a = entrenados(A)[lambda t: (t.model == model) & (t.m == M)]
        b = entrenados(B)[lambda t: (t.model == model) & (t.m == M)]
        ka, kb = int(a.unfolded.astype(bool).sum()), int(b.unfolded.astype(bool).sum())
        _, p_f = fisher_exact([[ka, len(a)-ka], [kb, len(b)-kb]], alternative='greater')
        p_mw = mannwhitneyu(a.mse_x, b.mse_x, alternative='less').pvalue
        res.append(dict(model=model, m=M, espectral=f'{ka}/{len(a)}', aleatorio=f'{kb}/{len(b)}',
                        fisher_p_espectral_mayor=p_f, mse_espectral=a.mse_x.median(), mse_aleatorio=b.mse_x.median(),
                        mannwhitney_p_espectral_menor=p_mw))
    res = pd.DataFrame(res)
    display(res.style.format({'fisher_p_espectral_mayor': '{:.3g}', 'mannwhitney_p_espectral_menor': '{:.3g}',
                              'mse_espectral': '{:.4g}', 'mse_aleatorio': '{:.4g}'})
            .set_caption(f'Prueba decisiva en m = {M}: mismos modos, misma clase, distinto camino'))
    ctrl = res[res.model == 'vae'].iloc[0]
    test = res[res.model == 'vae_warm'].iloc[0]
    print(f"Control (scratch, deberían ser iguales): Fisher p = {ctrl.fisher_p_espectral_mayor:.3g}"
          f"{'  <-- REVISAR: el control no debería salir significativo' if ctrl.fisher_p_espectral_mayor < .05 else ''}")
    print(f"Prueba (warm): Fisher p = {test.fisher_p_espectral_mayor:.3g} -> "
          + ('el ORDEN ESPECTRAL importa: el currículo es el mecanismo.' if test.fisher_p_espectral_mayor < .05 else
             'sin diferencia: el mérito es de la continuación en m, no del orden espectral.'))

In [ ]:
# ---- D: ¿el umbral de m sigue la posición del modo de altura? ----
Dv, Dd = cargar('D_30x10_vae'), cargar('D_30x10_direct')
if Dv is not None:
    tD = tabla_despliegue(Dv, 'D 30×10')
    display(tD.style.format(fmt).set_caption('D · 30×10, orden espectral'))
    display(pd.DataFrame({'oráculo 30×10': Dv[Dv.model == 'oracle'].groupby('m').chart_mse_z.median()})
            .style.format('{:.3g}'))
if Dd is not None:
    d = entrenados(Dd)[lambda t: t.model == 'direct']
    piso = 10**2/12/3
    s = d.groupby('m').mse_x.median()
    print(f'direct 30×10 (piso sin el modo de altura = Var(h)/3 = {piso:.3f}):')
    print(s.round(4).to_string())
    bajo = [int(m) for m, v in s.items() if v < piso/10]
    print('primer m con MSE_x < piso/10:', bajo[0] if bajo else 'ninguno', '| predicción: 3 o 4')

In [ ]:
# ---- Figura resumen para el paper (2x2) ----
# Autocontenida: solo lee los metrics.csv, así que puede re-ejecutarse sin repetir entrenamientos.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

WORK = Path(globals().get('WORK', '/kaggle/working'))

def _cargar(tag):
    csv = WORK / f'results_swiss_{tag}' / 'metrics.csv'
    return pd.read_csv(csv) if csv.exists() else None

def _cp(k, n, a=.05):
    lo = beta_dist.ppf(a/2, k, n-k+1) if k > 0 else 0.
    hi = beta_dist.ppf(1-a/2, k+1, n-k) if k < n else 1.
    return lo, hi

def _tasa(d, model):
    """Fracción desplegada por m.  Para vae_warm incluye m = m0: es el inicio de la cadena
    (fila duplicada del scratch, marcada warm_start); así la curva arranca donde arranca el camino."""
    g = d[(d.model == model) & d.m.notna() & d.unfolded.notna()]
    rows = []
    for m, q in g.groupby('m'):
        k, n = int(q.unfolded.astype(bool).sum()), len(q)
        lo, hi = _cp(k, n)
        rows.append((m, k/n, lo, hi, n))
    return pd.DataFrame(rows, columns=['m', 'tasa', 'lo', 'hi', 'n'])

def piso_prefijo(W, H, K):
    """Piso afín del oráculo (MSE por coordenada de la carta) cuando el prefijo contiene
    solo los armónicos de anchura k <= K y todavía no el modo de altura."""
    s = sum((4*W/(k*k*np.pi**2))**2 for k in range(K+1, 4001) if k % 2)
    return .5*(.5*s + H*H/12)

A, B, Dv = _cargar('A_spectral'), _cargar('B_shuffled'), _cargar('D_30x10_vae')
GEOM = {'60×10': dict(d=A, W=60., H=10., banda=(6, 7), c='C4', pred={2: 2, 4: 4}),
        '30×10': dict(d=Dv, W=30., H=10., banda=(3, 4), c='C2', pred={2: 2})}

fig, ax = plt.subplots(2, 2, figsize=(12.5, 8.6))
(a, b), (c, e) = ax

# (a) 60x10: orden espectral vs aleatorio, continuación vs desde cero
estilos = [(A, 'vae_warm', 'C4', '-', 'o', 'continuación, orden espectral'),
           (B, 'vae_warm', 'C1', '--', 's', 'continuación, orden aleatorio'),
           (A, 'vae', 'C0', ':', '^', 'desde cero, orden espectral'),
           (B, 'vae', 'C7', ':', 'v', 'desde cero, orden aleatorio')]
for i, (d, model, col, ls, mk, lab) in enumerate(estilos):
    if d is None:
        continue
    t = _tasa(d, model)
    a.errorbar(t.m*1.035**(i-1.5), t.tasa, yerr=[t.tasa-t.lo, t.hi-t.tasa], color=col, ls=ls, marker=mk, capsize=3, label=lab)
    q = d[(d.model == model) & d.m.notna()].groupby('m').mse_x.median()
    b.plot(q.index, q.values, color=col, ls=ls, marker=mk, label=lab)
a.set(xscale='log', ylim=(-.05, 1.05), xlabel='m (modos retenidos)', ylabel='fracción con carta desplegada',
      title='(a) 60×10: recuperación de la carta (IC 95 %)')
a.legend(fontsize=7, loc='center left')

# (b) 60x10: reconstrucción
if A is not None:
    b.axhline(A[A.model == 'chart_decoder'].mse_x.mean(), color='k', ls=':', lw=.8,
              label='decoder sobre la carta verdadera')
b.axhline(10**2/12/3, color='gray', ls='-.', lw=.8, label='piso ambiental sin el modo de altura, Var(h)/3')
b.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$ (mediana)', title='(b) 60×10: reconstrucción')
b.legend(fontsize=7)

# (c) información del prefijo: oráculo afín en ambas geometrías + pisos predichos
for lab, G in GEOM.items():
    d = G['d']
    if d is None:
        continue
    o = d[d.model == 'oracle'].groupby('m').chart_mse_z
    med, q1, q3 = o.median(), o.quantile(.25), o.quantile(.75)
    c.plot(med.index, med.values, color=G['c'], marker='o', label=f'{lab}: oráculo afín (mediana)')
    c.fill_between(med.index, q1.values, q3.values, color=G['c'], alpha=.2, lw=0)
    for m, K in G['pred'].items():
        c.scatter([m], [piso_prefijo(G['W'], G['H'], K)], marker='*', s=160, facecolor='none',
                  edgecolor=G['c'], lw=1.3, zorder=5)
    c.axvspan(*G['banda'], color=G['c'], alpha=.10)
c.scatter([], [], marker='*', s=160, facecolor='none', edgecolor='k', label='piso del teorema (sin el modo de altura)')
c.fill_between([], [], [], color='gray', alpha=.15, label='posición predicha del modo de altura')
c.set(xscale='log', yscale='log', xlabel='m', ylabel='chart MSE$_z$ del oráculo afín',
      title='(c) Información del prefijo: el piso cae donde predice el teorema')
c.legend(fontsize=7, loc='lower left')

# (d) despliegue en ambas geometrías: desde cero y continuación
for lab, G in GEOM.items():
    d = G['d']
    if d is None:
        continue
    for j, (model, ls, mk, nombre) in enumerate((('vae_warm', '-', 'o', 'continuación'), ('vae', ':', '^', 'desde cero'))):
        t = _tasa(d, model)
        shift = 1.035**((0 if lab == '60×10' else 2) + j - 1.5)   # separa las barras de error
        e.errorbar(t.m*shift, t.tasa, yerr=[t.tasa-t.lo, t.hi-t.tasa], color=G['c'], ls=ls, marker=mk, capsize=3,
                   label=f'{lab}, {nombre} (n = {int(t.n.max())})')
    e.axvspan(*G['banda'], color=G['c'], alpha=.10)
e.set(xscale='log', ylim=(-.05, 1.05), xlabel='m', ylabel='fracción con carta desplegada',
      title='(d) El despliegue empieza donde llega el modo de altura')
e.legend(fontsize=7, loc='center right')

for axis in ax.flat:
    ms = sorted(set(pd.concat([G['d'].m.dropna() for G in GEOM.values() if G['d'] is not None]).astype(int)))
    axis.set_xticks(ms)
    axis.set_xticklabels([str(m) for m in ms], fontsize=8)
    axis.minorticks_off()
fig.tight_layout()
fig.savefig(WORK / 'swiss_cierre.png', dpi=200)
fig.savefig(WORK / 'swiss_cierre.pdf')
plt.show()

print('Pisos predichos (chart MSE_z) vs oráculo medido (mediana):')
for lab, G in GEOM.items():
    if G['d'] is None:
        continue
    o = G['d'][G['d'].model == 'oracle'].groupby('m').chart_mse_z.median()
    for m, K in G['pred'].items():
        p = piso_prefijo(G['W'], G['H'], K)
        print(f'  {lab} m={m}: teorema {p:.3f} | medido {o.loc[m]:.3f} ({100*(o.loc[m]/p-1):+.1f} %)')


## Qué descargar

`/kaggle/working/results_swiss_A_spectral`, `results_swiss_B_shuffled`, `results_swiss_D_30x10_vae`, `results_swiss_D_30x10_direct`, `beta_sweep_swiss_30x10`, `swiss_cierre.png/.pdf` y `swiss_cierre_tablas.csv`. Cada `provenance.json` guarda la permutación de modos usada en B.